# NirmaanAI — EDA 02: NASA C-MAPSS Turbofan Engine Degradation
**Module**: Phase 6 (RUL Submodule) & Phase 13 (Factory Health Score)
**Dataset**: `DATASET/02_NASA_CMAPSS/raw/CMaps/`


In [ ]:
import os, sys
sys.path.append(r'C:/NIRMAAN AI')
import pandas as pd
import numpy as np

cmaps_dir = r'C:/NIRMAAN AI/DATASET/02_NASA_CMAPSS/raw/CMaps'
cols = ['unit', 'cycle', 'op1', 'op2', 'op3'] + [f's{i}' for i in range(1, 22)]
df_train = pd.read_csv(os.path.join(cmaps_dir, 'train_FD001.txt'), sep=r'\s+', names=cols)
print(f'Train FD001 shape: {df_train.shape}')
print(f'Number of turbofan engines: {df_train["unit"].nunique()}')


## 1. Engine Lifespan & RUL Trajectory Construction


In [ ]:
max_cycle = df_train.groupby('unit')['cycle'].max().reset_index()
max_cycle.columns = ['unit', 'max_cycle']
df_train = df_train.merge(max_cycle, on='unit')
df_train['RUL'] = df_train['max_cycle'] - df_train['cycle']
print('RUL Summary Statistics:\n', df_train['RUL'].describe())


## 2. Sensor Screening: Monotonic Drift vs Non-informative Sensors


In [ ]:
sensor_std = df_train[[f's{i}' for i in range(1, 22)]].std()
flat_sensors = sensor_std[sensor_std < 0.01].index.tolist()
informative_sensors = sensor_std[sensor_std >= 0.01].index.tolist()
print(f'Flat non-informative sensors in FD001 ({len(flat_sensors)}): {flat_sensors}')
print(f'Degradation-informative sensors ({len(informative_sensors)}): {informative_sensors}')


## 3. Key Findings for NirmaanAI Engine
1. Sensors s2, s3, s4, s7, s8, s11, s12, s15 exhibit clear monotonic drift tracking component wear.
2. Machine degradation follows piece-wise linear health curves suitable for composite health indexing in Phase 13.
